# CITRUS — sgACC single-subject functional connectivity

**Goal:** Evaluate sgACC (BA25) resting-state connectivity changes before and after TUS,  
comparing exp-focused vs con-defocused sessions across 4 timepoints.

**Current status:** first-level / single-subject pipeline validation for `sub-05`.  
This is **not group-level inference yet**.

**Pipeline (Nilearn-based, workshop style):**
1. Load BOLD + confounds (MEPrep / proc-pMEICA output)
2. Transform subject-specific sgACC/BA25 k-plan masks into MNI space
3. Extract bilateral sgACC seed time series from the subject-specific mask
4. Extract Harvard-Oxford network ROI time series
5. Compute Fisher-z transformed FC values
6. Visualise single-subject temporal changes across sessions and timepoints

---
> **To use on another subject:** add the subject ID in `SUBJECTS` and make sure the MEPrep outputs + k-plan sgACC masks exist.



## ── CONFIGURATION — change only this cell ──

In [1]:
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# CHANGE THESE PATHS TO MATCH YOUR DATA
# ─────────────────────────────────────────────────────────────
MEPREP_ROOT  = Path("/Volumes/Extreme SSD/THESIS-MSC/MEPrep output")
CITRUS_INPUT = Path("/Users/hoaithunguyen/Projects/Master-thesis/CITRUS/data/input")
OUT_DIR      = Path("/Users/hoaithunguyen/Projects/Master-thesis/CITRUS/derivatives/rs_fmri")
FSL_BIN      = Path("/Users/hoaithunguyen/fsl/bin")

# Subjects to process
SUBJECTS   = ["sub-05"]   # add others when MEPrep is done
SESSIONS   = ["ses-exp", "ses-con"]
TIMEPOINTS = ["preTUS15", "postTUS15", "postTUS30", "postTUS45"]

# Acquisition parameters
TR          = 1.50          # seconds
SMOOTH_FWHM = 6.0           # mm
BANDPASS    = (0.01, 0.10)  # Hz

# ─────────────────────────────────────────────────────────────
# Harvard-Oxford network nodes (L+R averaged for lateralised)
# ─────────────────────────────────────────────────────────────
# Midline cortical — single bilateral node (cortical atlas label index)
MIDLINE_NODES = {
    "Frontal_Medial":  25,   # Frontal Medial Cortex
    "Paracingulate":   28,   # Paracingulate Gyrus
    "ACC":             29,   # Cingulate Gyrus, anterior division
    "PCC":             30,   # Cingulate Gyrus, posterior division
    "Precuneus":       31,   # Precuneous Cortex
}

# Lateralised — Left index, Right index (subcortical atlas)
LATERAL_NODES = {
    "Insula":      (2,  2),    # Insular Cortex (cortical — same label covers both hemi)
    "Hippocampus": (9,  19),   # Left/Right Hippocampus (subcortical)
    "Amygdala":    (10, 20),   # Left/Right Amygdala (subcortical)
    "Thalamus":    (4,  15),   # Left/Right Thalamus (subcortical)
    "Accumbens":   (11, 21),   # Left/Right Accumbens (subcortical)
}
# Note: Insula uses cortical atlas but label covers both hemispheres — will split by x-coord

# ─────────────────────────────────────────────────────────────
# Derived — do not change below
BOLD_PROC  = "pmeica"
BOLD_SPACE = "MNI152NLin2009cAsym"
NET_ROIS   = list(MIDLINE_NODES.keys()) + list(LATERAL_NODES.keys())

SES_COLORS = {"ses-exp": "#e05c5c", "ses-con": "#5c7de0"}
SES_LABELS = {"ses-exp": "Experimental (focused)", "ses-con": "Control (defocused)"}
TP_LABELS  = {"preTUS15": "Pre\n(−15 min)", "postTUS15": "Post\n(+15 min)",
              "postTUS30": "Post\n(+30 min)", "postTUS45": "Post\n(+45 min)"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Config ready.")
print("  SSD mounted:      ", MEPREP_ROOT.exists())
print("  CITRUS input dir: ", CITRUS_INPUT.exists())
print("  FSL flirt:        ", (FSL_BIN / "flirt").exists())

Config ready.
  SSD mounted:       True
  CITRUS input dir:  True
  FSL flirt:         True


## 0 — Imports

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from nilearn import image, plotting
from nilearn.maskers import NiftiMasker

print("All imports OK")

All imports OK


## 1 — Helper: locate files

In [3]:
import subprocess
import os
import numpy as np
import nibabel as nib
from nilearn import image
from pathlib import Path

FSLDIR = FSL_BIN.parent

def _flirt(args):
    """Run flirt with FSL env vars set explicitly."""
    fsl_env = {**os.environ,
               "FSLDIR": str(FSLDIR),
               "FSLOUTPUTTYPE": "NIFTI_GZ",
               "PATH": str(FSL_BIN) + ":" + os.environ.get("PATH", "")}
    return subprocess.run([str(FSL_BIN / "flirt")] + args,
                          capture_output=True, text=True, env=fsl_env)


def register_kplan_to_mni(sub: str):
    """Register kplan T1w -> MNI T1w via flirt. Returns .mat path or None."""
    kplan_t1 = CITRUS_INPUT / sub / f"{sub}_T1w_kplan.nii.gz"
    mni_t1   = MEPREP_ROOT / "ses-intake" / f"{sub}_ses-intake_acq-HCP_space-{BOLD_SPACE}_desc-preproc_T1w.nii.gz"
    out_dir  = OUT_DIR / "seeds" / sub
    out_dir.mkdir(parents=True, exist_ok=True)
    mat_path = out_dir / f"{sub}_kplan_to_MNI.mat"

    if mat_path.exists():
        print(f"  [{sub}] flirt .mat already exists — skipping")
        return mat_path
    if not kplan_t1.exists():
        print(f"  [{sub}] MISSING kplan T1w: {kplan_t1}")
        return None
    if not mni_t1.exists():
        print(f"  [{sub}] MISSING MNI T1w: {mni_t1}")
        return None

    print(f"  [{sub}] Running flirt (kplan T1w -> MNI T1w)...")
    result = _flirt(["-in", str(kplan_t1), "-ref", str(mni_t1),
                     "-omat", str(mat_path), "-dof", "12",
                     "-cost", "corratio", "-interp", "spline"])
    if result.returncode != 0:
        print(f"  [{sub}] flirt FAILED:\n{result.stderr}")
        return None
    print(f"  [{sub}] flirt done -> {mat_path.name}")
    return mat_path


def warp_mask_to_mni(sub: str, hemi: str, mat_path) -> Path | None:
    """Warp one hemisphere sgACC mask to MNI. Returns output mask path or None."""
    mask_kplan = CITRUS_INPUT / sub / f"sgACC_BA25_{hemi}_kplan.nii.gz"
    mni_t1     = MEPREP_ROOT / "ses-intake" / f"{sub}_ses-intake_acq-HCP_space-{BOLD_SPACE}_desc-preproc_T1w.nii.gz"
    out_dir    = OUT_DIR / "seeds" / sub
    out_mask   = out_dir / f"{sub}_sgACC_BA25_{hemi}_MNI.nii.gz"

    if out_mask.exists():
        return out_mask

    if not mask_kplan.exists():
        print(f"  [{sub}] MISSING kplan mask: {mask_kplan}")
        return None
    if not mni_t1.exists():
        print(f"  [{sub}] MISSING MNI T1w (SSD not mounted?): {mni_t1}")
        return None

    result = _flirt(["-in", str(mask_kplan), "-ref", str(mni_t1),
                     "-init", str(mat_path), "-applyxfm",
                     "-interp", "nearestneighbour", "-out", str(out_mask)])
    if result.returncode != 0:
        print(f"  [{sub}] mask warp FAILED:\n{result.stderr}")
        return None
    if not out_mask.exists():
        print(f"  [{sub}] mask warp produced no output:\n{result.stderr}")
        return None
    return out_mask


def make_bilateral_seed_mask(sub: str, mask_L: Path, mask_R: Path) -> Path | None:
    """Combine L+R warped sgACC masks into one bilateral voxel mask."""
    out_dir  = OUT_DIR / "seeds" / sub
    out_path = out_dir / f"{sub}_sgACC_BA25_bilateral_MNI.nii.gz"
    if out_path.exists():
        return out_path
    img_L = nib.load(str(mask_L))
    img_R = nib.load(str(mask_R))
    bilateral = (img_L.get_fdata() + img_R.get_fdata()) > 0.5
    nib.save(nib.Nifti1Image(bilateral.astype(np.float32), img_L.affine), str(out_path))
    return out_path


# ── Run for all subjects ──────────────────────────────────────
SUBJECT_MASKS = {}   # sub -> Path to bilateral sgACC MNI mask

for sub in SUBJECTS:
    mat = register_kplan_to_mni(sub)
    if mat is None:
        continue

    hemi_masks = {}
    for hemi in ["L", "R"]:
        p = warp_mask_to_mni(sub, hemi, mat)
        if p is not None:
            hemi_masks[hemi] = p
            n_vox = int((nib.load(str(p)).get_fdata() > 0.5).sum())
            print(f"  [{sub}] sgACC_{hemi} MNI mask: {n_vox} voxels")

    if "L" in hemi_masks and "R" in hemi_masks:
        bil = make_bilateral_seed_mask(sub, hemi_masks["L"], hemi_masks["R"])
    elif hemi_masks:
        bil = list(hemi_masks.values())[0]
    else:
        print(f"  [{sub}] no masks available — skipping")
        continue

    SUBJECT_MASKS[sub] = bil
    n_vox = int((nib.load(str(bil)).get_fdata() > 0.5).sum())
    print(f"  [{sub}] bilateral seed mask: {n_vox} voxels -> {bil.name}")

print(f"\nSeed masks ready for {len(SUBJECT_MASKS)} subject(s):")
for sub, p in SUBJECT_MASKS.items():
    print(f"  {sub}: {p}")


  [sub-05] flirt .mat already exists — skipping
  [sub-05] sgACC_L MNI mask: 570 voxels
  [sub-05] sgACC_R MNI mask: 812 voxels
  [sub-05] bilateral seed mask: 1382 voxels -> sub-05_sgACC_BA25_bilateral_MNI.nii.gz

Seed masks ready for 1 subject(s):
  sub-05: /Users/hoaithunguyen/Projects/Master-thesis/CITRUS/derivatives/rs_fmri/seeds/sub-05/sub-05_sgACC_BA25_bilateral_MNI.nii.gz


## 1b — Per-subject sgACC seed mask in MNI space
Warp each subject's kplan sgACC masks (L + R hemispheres) to MNI space using FSL flirt.
Merge L + R into one bilateral voxel mask — all sgACC voxels from both hemispheres.
This mask is used directly as the seed: mean BOLD signal across all its voxels = the sgACC time series.

In [4]:
def find_run(sub, ses, acq):
    """Return (bold_path, mask_path, confounds_path) or (None,None,None)."""
    func_dir = MEPREP_ROOT / ses / "func"
    stem = f"{sub}_{ses}_task-rest_acq-{acq}_proc-{BOLD_PROC}"
    bold = func_dir / f"{stem}_space-{BOLD_SPACE}_desc-preproc_bold.nii.gz"
    mask = func_dir / f"{stem}_space-{BOLD_SPACE}_desc-brain_mask.nii.gz"
    conf = func_dir / f"{stem}_desc-confounds_timeseries.tsv"
    if bold.exists() and mask.exists() and conf.exists():
        return bold, mask, conf
    print(f"  MISSING: {sub} | {ses} | {acq}")
    return None, None, None


def get_confounds(conf_path):
    """Build confound matrix: 24 HMP + 5 aCompCor + cosine drift + motion outliers."""
    df = pd.read_csv(conf_path, sep="\t")
    hmp = ["trans_x","trans_y","trans_z","rot_x","rot_y","rot_z"]
    hmp24 = hmp + [f"{c}_derivative1" for c in hmp] + \
                  [f"{c}_power2"       for c in hmp] + \
                  [f"{c}_derivative1_power2" for c in hmp]
    acomp  = [f"a_comp_cor_{i:02d}" for i in range(5)]
    cosine = [c for c in df.columns if c.startswith("cosine")]
    outliers = [c for c in df.columns if c.startswith("motion_outlier")]
    cols = [c for c in hmp24 + acomp + cosine + outliers if c in df.columns]
    return df[cols].fillna(0).values


# Quick check: can we find sub-05 data?
bold, mask, conf = find_run("sub-05", "ses-exp", "preTUS15")
print("Found:", bold.name if bold else "NOT FOUND")

Found: sub-05_ses-exp_task-rest_acq-preTUS15_proc-pmeica_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz


## 2 — Seed-based FC (nilearn style)

Following the [nilearn seed-to-voxel tutorial](https://nilearn.github.io/stable/auto_examples/03_connectivity/plot_seed_to_voxel_correlation.html).  
Confound regression + bandpass happens **inside** the masker — no separate preprocessing step needed.

In [5]:
def compute_seed_fc_mask(bold_path, mask_path, conf_path, seed_mask_path):
    """
    Seed-based whole-brain FC map using a subject-specific sgACC voxel mask.

    This replaces the older 6-mm coordinate-sphere seed version, because the thesis
    target is subject-specific sgACC/BA25. It returns a Fisher-z whole-brain map and
    the extracted sgACC seed time series.
    """
    confounds = get_confounds(conf_path)

    # Resample the subject-specific seed mask to the current BOLD grid.
    bold_ref = image.index_img(image.load_img(str(bold_path)), 0)
    seed_img = image.resample_to_img(
        image.load_img(str(seed_mask_path)), bold_ref, interpolation="nearest"
    )

    if int((seed_img.get_fdata() > 0).sum()) == 0:
        raise ValueError(f"Seed mask is empty after resampling: {seed_mask_path}")

    # Extract sgACC seed time series.
    seed_masker = NiftiMasker(
        mask_img=seed_img,
        smoothing_fwhm=None,  # avoid blurring the small seed into surrounding tissue
        detrend=True,
        standardize="zscore_sample",
        low_pass=BANDPASS[1],
        high_pass=BANDPASS[0],
        t_r=TR,
        verbose=0,
    )
    seed_voxel_ts = seed_masker.fit_transform(str(bold_path), confounds=confounds)
    seed_ts = seed_voxel_ts.mean(axis=1)

    # Extract whole-brain voxel time series.
    brain_masker = NiftiMasker(
        mask_img=str(mask_path),
        smoothing_fwhm=SMOOTH_FWHM,
        detrend=True,
        standardize="zscore_sample",
        low_pass=BANDPASS[1],
        high_pass=BANDPASS[0],
        t_r=TR,
        verbose=0,
    )
    brain_ts = brain_masker.fit_transform(str(bold_path), confounds=confounds)

    # Pearson correlation with seed; Fisher z-transform.
    r = np.corrcoef(brain_ts.T, seed_ts)[-1, :-1]
    z = np.arctanh(np.clip(r, -0.999, 0.999))

    z_img = brain_masker.inverse_transform(z)
    return z_img, seed_ts


print("compute_seed_fc_mask() defined")



compute_seed_fc_mask() defined


### 2a — Run for all subjects × sessions × timepoints × seeds

In [ ]:
fc_results = {}   # key: (sub, ses, acq, "sgACC_BA25_mask") -> z_img

for sub in SUBJECTS:
    seed_mask = SUBJECT_MASKS.get(sub)
    if seed_mask is None:
        continue

    for ses in SESSIONS:
        for acq in TIMEPOINTS:
            bold, mask, conf = find_run(sub, ses, acq)
            if bold is None:
                continue

            seed_name = "sgACC_BA25_mask"
            key = (sub, ses, acq, seed_name)
            out_path = OUT_DIR / "seed_fc" / sub / ses / \
                           f"{sub}_{ses}_acq-{acq}_seed-{seed_name}_fc.nii.gz"

            if out_path.exists():
                fc_results[key] = nib.load(str(out_path))
                print(f" Loaded cache: {sub} | {ses} | {acq} | {seed_name}")
                continue

            print(f" Computing whole-brain FC: {sub} | {ses} | {acq} | {seed_name} ...", end=" ")
            z_img, _ = compute_seed_fc_mask(bold, mask, conf, seed_mask)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            z_img.to_filename(str(out_path))
            fc_results[key] = z_img
            print("done")

print(f"Total whole-brain FC maps computed/loaded: {len(fc_results)}")



 Computing whole-brain FC: sub-05 | ses-exp | preTUS15 | sgACC_BA25_mask ... 

### 2b — Visualise FC maps (workshop style)

In [ ]:
from nilearn import datasets, image
import numpy as np

# ── Build Harvard-Oxford network masks ───────────────────────
print("Loading Harvard-Oxford atlases...")
ho_cort = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
ho_sub  = datasets.fetch_atlas_harvard_oxford("sub-maxprob-thr25-2mm")

cort_img = ho_cort.maps
sub_img  = ho_sub.maps

# Use one available BOLD image as the reference grid for all ROI masks.
bold_ref = None
for sub in SUBJECTS:
    for ses in SESSIONS:
        for acq in TIMEPOINTS:
            bold, mask, conf = find_run(sub, ses, acq)
            if bold is not None:
                bold_ref = bold
                break
        if bold_ref is not None:
            break
    if bold_ref is not None:
        break

if bold_ref is None:
    raise FileNotFoundError("No BOLD file found. Check MEPREP_ROOT, SUBJECTS, SESSIONS, and TIMEPOINTS.")

ref_img = image.index_img(image.load_img(str(bold_ref)), 0)

# Resample atlases to BOLD grid.
cort_res = image.resample_to_img(cort_img, ref_img, interpolation="nearest")
sub_res  = image.resample_to_img(sub_img,  ref_img, interpolation="nearest")
cort_data = cort_res.get_fdata()
sub_data  = sub_res.get_fdata()

def make_roi_mask(label_img_data, label_idx, ref_img):
    """Binary mask for a given atlas label index."""
    mask = (label_img_data == label_idx).astype(np.float32)
    return image.new_img_like(ref_img, mask)

def make_bilateral_mask(data, l_idx, r_idx, ref_img):
    """Combine left + right atlas labels into one bilateral mask."""
    mask = ((data == l_idx) | (data == r_idx)).astype(np.float32)
    return image.new_img_like(ref_img, mask)

# ── Build network ROI masks ───────────────────────────────────
NETWORK_MASKS = {}

for name, idx in MIDLINE_NODES.items():
    NETWORK_MASKS[name] = make_roi_mask(cort_data, idx, ref_img)
    print(f"  {name:20s} (cortical label {idx:3d}): {int((cort_data == idx).sum())} voxels")

# Insula is a cortical bilateral label in this atlas.
insula_idx = LATERAL_NODES["Insula"][0]
NETWORK_MASKS["Insula"] = make_roi_mask(cort_data, insula_idx, ref_img)
print(f"  {'Insula':20s} (cortical label {insula_idx:3d}, bilateral): {int((cort_data == insula_idx).sum())} voxels")

for name, (l_idx, r_idx) in LATERAL_NODES.items():
    if name == "Insula":
        continue
    NETWORK_MASKS[name] = make_bilateral_mask(sub_data, l_idx, r_idx, ref_img)
    print(f"  {name:20s} (subcortical L{l_idx}+R{r_idx}): {int(((sub_data == l_idx) | (sub_data == r_idx)).sum())} voxels")

# Use the actual mask order for later plots.
NET_ROIS = list(NETWORK_MASKS.keys())
print(f"
Network has {len(NETWORK_MASKS)} nodes: {NET_ROIS}")



## 3 — ROI network connectivity

Extract time series from 12 sgACC network nodes and compute pairwise connectivity matrix.

In [ ]:
def extract_network_connectivity(bold_path, mask_path, conf_path, seed_mask=None):
    """
    Compute sgACC seed -> network ROI FC using voxel masks.

    Returns
    -------
    z_mat     : ROI-to-ROI Fisher-z matrix among the Harvard-Oxford network nodes
    roi_ts    : (n_TRs, n_rois) ROI mean time series
    roi_names : list of ROI names
    seed_fc   : {roi_name: Fisher-z sgACC seed-to-ROI value}, or None if no seed mask
    """
    confounds = get_confounds(conf_path)
    roi_names = list(NETWORK_MASKS.keys())

    # ── Network ROI time series ───────────────────────────────
    roi_ts_list = []
    for roi_name, roi_mask in NETWORK_MASKS.items():
        masker = NiftiMasker(
            mask_img=roi_mask,
            smoothing_fwhm=SMOOTH_FWHM,
            detrend=True,
            standardize="zscore_sample",
            low_pass=BANDPASS[1],
            high_pass=BANDPASS[0],
            t_r=TR,
            verbose=0,
        )
        ts = masker.fit_transform(str(bold_path), confounds=confounds)
        if ts.shape[1] == 0:
            raise ValueError(f"ROI mask has zero voxels after masking: {roi_name}")
        roi_ts_list.append(ts.mean(axis=1))

    roi_ts = np.column_stack(roi_ts_list)

    # ── ROI-to-ROI connectivity matrix ────────────────────────
    r_mat = np.corrcoef(roi_ts.T)
    z_mat = np.arctanh(np.clip(r_mat, -0.999, 0.999))
    np.fill_diagonal(z_mat, 0)

    # ── Subject-specific sgACC seed -> each network ROI ───────
    seed_fc = None
    if seed_mask is not None:
        bold_ref = image.index_img(image.load_img(str(bold_path)), 0)
        seed_img = image.resample_to_img(
            image.load_img(str(seed_mask)), bold_ref, interpolation="nearest"
        )
        if int((seed_img.get_fdata() > 0).sum()) == 0:
            raise ValueError(f"Seed mask is empty after resampling: {seed_mask}")

        seed_masker = NiftiMasker(
            mask_img=seed_img,
            smoothing_fwhm=None,
            detrend=True,
            standardize="zscore_sample",
            low_pass=BANDPASS[1],
            high_pass=BANDPASS[0],
            t_r=TR,
            verbose=0,
        )
        seed_ts = seed_masker.fit_transform(str(bold_path), confounds=confounds).mean(axis=1)

        seed_fc = {}
        for i, roi_name in enumerate(roi_names):
            r = np.corrcoef(seed_ts, roi_ts[:, i])[0, 1]
            seed_fc[roi_name] = float(np.arctanh(np.clip(r, -0.999, 0.999)))

    return z_mat, roi_ts, roi_names, seed_fc


# ── Run for all subjects / sessions / timepoints ──────────────
net_records   = []
conn_matrices = {}
ROI_NAMES     = None

for sub in SUBJECTS:
    seed_mask = SUBJECT_MASKS.get(sub)
    if seed_mask is None:
        print(f"  WARNING: no seed mask for {sub}; seed-to-ROI FC will not be computed")

    for ses in SESSIONS:
        for acq in TIMEPOINTS:
            bold, mask, conf = find_run(sub, ses, acq)
            if bold is None:
                continue

            print(f"  Network: {sub} | {ses} | {acq} ...", end=" ")
            z_mat, roi_ts, roi_names, seed_fc = extract_network_connectivity(
                bold, mask, conf, seed_mask=seed_mask
            )

            if ROI_NAMES is None:
                ROI_NAMES = roi_names
            conn_matrices[(sub, ses, acq)] = z_mat

            if seed_fc is not None:
                for roi_name, fc_z in seed_fc.items():
                    net_records.append({
                        "subject": sub,
                        "session": ses,
                        "timepoint": acq,
                        "roi": roi_name,
                        "fc_z": fc_z,
                    })
            print("done")

df_fc = pd.DataFrame(net_records)
print(f"
FC table: {df_fc.shape}  columns: {df_fc.columns.tolist()}")
if df_fc.empty:
    print("WARNING: df_fc is empty — check SUBJECT_MASKS and find_run output above")
else:
    print(df_fc.head())



### 3a — Visualise connectivity matrix

In [ ]:
sub = SUBJECTS[0]
n_tp = len(TIMEPOINTS)

fig, axes = plt.subplots(len(SESSIONS), n_tp, figsize=(5*n_tp, 9),
                         facecolor="white")
fig.suptitle(f"{sub} — sgACC network connectivity matrix (Fisher z)",
             fontsize=13, fontweight="bold")

for r, ses in enumerate(SESSIONS):
    for c, acq in enumerate(TIMEPOINTS):
        ax = axes[r, c]
        key = (sub, ses, acq)
        if key not in conn_matrices:
            ax.set_visible(False)
            continue
        im = ax.imshow(conn_matrices[key], cmap="RdBu_r",
                       vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(len(ROI_NAMES)))
        ax.set_xticklabels(ROI_NAMES, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(ROI_NAMES)))
        ax.set_yticklabels(ROI_NAMES, fontsize=7)
        if r == 0:
            ax.set_title(TP_LABELS[acq], fontsize=10)
        if c == 0:
            ax.set_ylabel(SES_LABELS[ses], fontsize=9, fontweight="bold")

fig.colorbar(im, ax=axes, shrink=0.5, label="Fisher z", pad=0.02)
plt.savefig(str(OUT_DIR / f"{sub}_connectivity_matrix.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 4 — Temporal line plots: sgACC → each network node

In [ ]:
n_rois = len(NET_ROIS)
n_cols = 4
n_rows = int(np.ceil(n_rois / n_cols))
TP_X   = list(range(len(TIMEPOINTS)))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows), facecolor="white")
fig.suptitle("Single-subject sgACC connectivity to network nodes over time
(exp-focused vs con-defocused)",
             fontsize=13, fontweight="bold")
axes = axes.flatten()

for ax_i, roi in enumerate(NET_ROIS):
    ax = axes[ax_i]
    roi_df = df_fc[df_fc["roi"] == roi]

    for ses in SESSIONS:
        ses_df = roi_df[roi_df["session"] == ses]
        vals = [ses_df[ses_df["timepoint"] == tp]["fc_z"].mean() for tp in TIMEPOINTS]
        color = SES_COLORS[ses]
        ax.plot(TP_X, vals, "o-", color=color, lw=2.5, label=SES_LABELS[ses], zorder=3)

    ax.axhline(0, color="gray", lw=0.8, ls="--", alpha=0.5)
    ax.axvspan(-0.5, 0.5, color="lightgray", alpha=0.2, label="pre-TUS")
    ax.set_xticks(TP_X)
    ax.set_xticklabels([TP_LABELS[t] for t in TIMEPOINTS], fontsize=8)
    ax.set_title(roi, fontsize=11, fontweight="bold")
    ax.set_ylabel("FC (Fisher z)", fontsize=9)
    ax.spines[["top","right"]].set_visible(False)
    if ax_i == 0:
        ax.legend(fontsize=8, loc="upper right")

for ax in axes[n_rois:]:
    ax.set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(str(OUT_DIR / "sgacc_temporal_fc_single_subject.png"), dpi=150, bbox_inches="tight")
plt.show()



## 5 — Radar chart: sgACC connectivity profile

In [ ]:
import numpy as np

N = len(NET_ROIS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

# ── Single-subject radar per timepoint ───────────────────────
fig, axes = plt.subplots(1, len(TIMEPOINTS), figsize=(20, 5),
                         subplot_kw={"projection": "polar"}, facecolor="white")
fig.suptitle("Single-subject sgACC network connectivity profile",
             fontsize=13, fontweight="bold", y=1.03)

r_max = df_fc["fc_z"].abs().quantile(0.95) * 1.2 if len(df_fc) > 0 else 1.0
if not np.isfinite(r_max) or r_max == 0:
    r_max = 1.0

for ax, tp in zip(axes, TIMEPOINTS):
    for ses in SESSIONS:
        vals = [df_fc[(df_fc["session"]==ses) & (df_fc["timepoint"]==tp) &
                      (df_fc["roi"]==roi)]["fc_z"].mean() for roi in NET_ROIS]
        vals_closed = vals + vals[:1]
        color = SES_COLORS[ses]
        ax.plot(angles, vals_closed, "-o", color=color, lw=2,
                markersize=4, label=SES_LABELS[ses])
        ax.fill(angles, vals_closed, color=color, alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(NET_ROIS, fontsize=8)
    ax.set_ylim(-r_max, r_max)
    ax.plot(angles, [0]*len(angles), "-", color="black", lw=0.8, alpha=0.4)
    ax.spines["polar"].set_visible(False)
    ax.grid(color="gray", lw=0.4, alpha=0.3)
    ax.set_title(TP_LABELS[tp], fontsize=10, fontweight="bold", pad=14)

axes[0].legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=9, frameon=False)
plt.savefig(str(OUT_DIR / "sgacc_radar_single_subject.png"), dpi=150, bbox_inches="tight")
plt.show()



In [ ]:
# ── Δ FC radar (post − pre) ──
post_tps = [tp for tp in TIMEPOINTS if tp != "preTUS15"]
fig, axes = plt.subplots(1, len(post_tps), figsize=(16, 5),
                         subplot_kw={"projection": "polar"}, facecolor="white")
fig.suptitle("Δ sgACC connectivity from pre-TUS baseline",
             fontsize=13, fontweight="bold", y=1.03)

for ax, tp in zip(axes, post_tps):
    for ses in SESSIONS:
        pre  = [df_fc[(df_fc["session"]==ses) & (df_fc["timepoint"]=="preTUS15") &
                      (df_fc["roi"]==roi)]["fc_z"].mean() for roi in NET_ROIS]
        post = [df_fc[(df_fc["session"]==ses) & (df_fc["timepoint"]==tp) &
                      (df_fc["roi"]==roi)]["fc_z"].mean() for roi in NET_ROIS]
        delta = [p - pr for p, pr in zip(post, pre)]
        delta_closed = delta + delta[:1]
        color = SES_COLORS[ses]
        ax.plot(angles, delta_closed, "-o", color=color, lw=2,
                markersize=4, label=SES_LABELS[ses])
        ax.fill(angles, delta_closed, color=color, alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(NET_ROIS, fontsize=8)
    ax.set_ylim(-0.6, 0.6)
    ax.set_yticks([-0.3, 0, 0.3])
    ax.set_yticklabels(["-0.3", "0", "0.3"], fontsize=7, color="gray")
    ax.plot(angles, [0]*len(angles), "-", color="black", lw=1, alpha=0.5)
    ax.spines["polar"].set_visible(False)
    ax.grid(color="gray", lw=0.4, alpha=0.3)
    ax.set_title(f"Δ {TP_LABELS[tp]}", fontsize=10, fontweight="bold", pad=14)

axes[0].legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=9, frameon=False)
plt.savefig(str(OUT_DIR / "sgacc_radar_delta.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6 — Single-subject summary table


In [ ]:
if len(df_fc) > 0:
    # For sub-05 only, each cell is one observed FC value, not a group mean.
    summary = df_fc.sort_values(["roi", "session", "timepoint"]).copy()
    pivot = summary.pivot_table(
        index="roi", columns=["session", "timepoint"], values="fc_z", aggfunc="first"
    ).round(3)
    print(pivot.to_string())

    summary.to_csv(str(OUT_DIR / "sgacc_fc_single_subject_values.tsv"), sep="	", index=False)
    print(f"
Saved -> {OUT_DIR}/sgacc_fc_single_subject_values.tsv")
else:
    print("No data yet — run cells above first.")



---
## Notes

### Current status
- Only `sub-05` has currently been processed through the complete FC pipeline.
- Current outputs should be interpreted as first-level single-subject results only.
- Group-level inference will only be performed after additional subjects have completed MEPrep preprocessing and have been added to `SUBJECTS`.

### Confound strategy
- 24 head-motion parameters / HMP: 6 motion parameters + derivatives + squared terms
- 5 aCompCor components
- Cosine drift regressors
- Motion outlier regressors, when available
- No additional WM/CSF signal regression is currently applied
- Physiological noise reduction is primarily handled through MEICA/pMEICA denoising
- Band-pass filtering: 0.01-0.10 Hz, applied inside the Nilearn maskers

### sgACC seed definition
- The primary seed is a subject-specific bilateral BA25/sgACC voxel mask, not a 6 mm coordinate sphere.
- Left and right sgACC masks are generated in subject anatomical/k-plan space.
- Masks are transformed to MNI space using FSL FLIRT.
- Left and right masks are merged into one bilateral sgACC seed mask.
- The mean BOLD signal is extracted from all voxels inside this bilateral sgACC mask using `NiftiMasker`.
- `NiftiSpheresMasker` is intentionally not used in the current primary pipeline.

### To use on additional subjects
1. Change paths in the `CONFIGURATION` cell.
2. Update the `SUBJECTS` list.
3. Verify availability of the subject-specific sgACC masks and MEPrep outputs.
4. Run all cells top to bottom (`Kernel -> Restart & Run All`).
